In [18]:
# %% Cell 1 — Imports & setup
import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR  = Path("../public")
OUT_DIR.mkdir(exist_ok=True)

XLSX =  "Feb26CBOProjections.xlsx"

xl = pd.ExcelFile(XLSX)
print("Sheets:", xl.sheet_names)

# %% Cell 2 — Helpers
def clean(v):
    if isinstance(v, float) and np.isnan(v):
        return None
    s = str(v).strip().strip("'").strip()
    return s if s not in ("", "*", "—", "n.a.", " ") else None

def to_num(v):
    c = clean(v)
    if c is None:
        return None
    try:
        return float(str(c).replace(",", ""))
    except:
        return None

def is_year(v):
    c = clean(v)
    if c is None:
        return False
    try:
        y = int(float(c))
        return 2024 <= y <= 2037
    except:
        return False

# %% Cell 3 — Raw inspection helper
def peek(sheet, rows=range(0, 25)):
    raw = pd.read_excel(XLSX, sheet_name=sheet, header=None)
    print(f"=== {sheet} ({len(raw)} rows x {len(raw.columns)} cols) ===")
    for i in rows:
        if i >= len(raw):
            break
        row = raw.iloc[i]
        vals = [repr(v) for v in row if not (isinstance(v, float) and np.isnan(v))]
        if vals:
            print(f"  row {i:2d}: {vals}")

Sheets: ['Contents', 'Table 1-1', 'Table 1-2', 'Table 1-3', 'Table 1-4', 'Table 3-1', 'Table 3-2', 'Table 3-2, unadj', 'Table 3-3', 'Table 3-4', 'Table 3-5', 'Table 3-6', 'Table 3-7', 'Table 3-8', 'Box 3-2 Table', 'Table 5-1']


In [19]:
# %% Cell 4 — Parse Table 1-1 by explicit row index
# Structure (confirmed from peek):
#   row  8: year headers (cols 1-12 = 2025-2036, cols 13-14 = summary totals → skip)
#   row 16: Total Revenues
#   row 20: Mandatory Outlays
#   row 21: Discretionary Outlays
#   row 22: Net Interest
#   row 23: Total Outlays
#   rows 11-15: individual revenue line items
# "On-budget"/"Off-budget" rows skipped (not needed)

def parse_table_1_1(xlsx_path):
    raw = pd.read_excel(xlsx_path, sheet_name="Table 1-1", header=None)

    # Build col -> year mapping from row 8, skipping summary cols
    col_years = {}
    for ci, v in enumerate(raw.iloc[8]):
        if is_year(v):
            col_years[ci] = int(float(clean(v)))

    def get_row(row_idx):
        row = raw.iloc[row_idx]
        return {yr: to_num(row.iloc[ci]) for ci, yr in col_years.items()}

    data = {
        "Individual Income Taxes": get_row(11),
        "Payroll Taxes":           get_row(12),
        "Corporate Income Taxes":  get_row(13),
        "Customs Duties":          get_row(14),
        "Other Revenues":          get_row(15),
        "Total Revenues":          get_row(16),
        "Mandatory Outlays":       get_row(20),
        "Discretionary Outlays":   get_row(21),
        "Net Interest":            get_row(22),
        "Total Outlays":           get_row(23),
        "Total Deficit":           get_row(26),   # negative in source — abs() when using
        "Debt Held by Public":     get_row(30),
        "GDP":                     get_row(32),   # Addendum row, past "Addendum:" header
    }

    return sorted(col_years.values()), data

years_11, data_11 = parse_table_1_1(XLSX)

print(f"Years: {years_11}")
print("\nSpot checks (2025 / 2026):")
checks = {
    "Total Revenues":    (5234.6,   5595.9),
    "Total Outlays":     (7009.986, 7448.619),
    "Net Interest":      (969.938,  1038.976),
    "Mandatory Outlays": (4167.623, 4529.339),
    "Total Deficit":     (-1775.37, -1852.703),  # negative
    "Debt Held by Public":(30172.402,32095.165),
    "GDP":               (30362.025,31902.006),
}
for label, (exp25, exp26) in checks.items():
    v25 = data_11[label][2025]
    v26 = data_11[label][2026]
    ok25 = "✓" if abs(v25 - exp25) < 0.1 else "✗"
    ok26 = "✓" if abs(v26 - exp26) < 0.1 else "✗"
    print(f"  {label:25s} 2025: {v25:.1f} {ok25}  2026: {v26:.1f} {ok26}")


Years: [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036]

Spot checks (2025 / 2026):
  Total Revenues            2025: 5234.6 ✓  2026: 5595.9 ✓
  Total Outlays             2025: 7010.0 ✓  2026: 7448.6 ✓
  Net Interest              2025: 969.9 ✓  2026: 1039.0 ✓
  Mandatory Outlays         2025: 4167.6 ✓  2026: 4529.3 ✓
  Total Deficit             2025: -1775.4 ✓  2026: -1852.7 ✓
  Debt Held by Public       2025: 30172.4 ✓  2026: 32095.2 ✓
  GDP                       2025: 30362.0 ✓  2026: 31902.0 ✓


In [20]:
peek("Table 1-3")

=== Table 1-3 (39 rows x 13 cols) ===
  row  0: ["'This file presents data that supplement CBO’s February 2026 report The Budget and Economic Outlook: 2026 to 2036.'"]
  row  1: ["'www.cbo.gov/publication/61882'"]
  row  4: ['"Table 1-3. \\nCBO\'s Baseline Projections of Federal Debt"']
  row  5: ["'Billions of dollars'"]
  row  7: ["'Actual, \\n2025'", 'np.float64(2026.0)', 'np.float64(2027.0)', 'np.float64(2028.0)', 'np.float64(2029.0)', 'np.float64(2030.0)', 'np.float64(2031.0)', 'np.float64(2032.0)', 'np.float64(2033.0)', 'np.float64(2034.0)', 'np.float64(2035.0)', 'np.float64(2036.0)']
  row  8: ["'Debt held by the public at the \\nbeginning of the year'", '28195.575', 'np.float64(30172.402)', 'np.float64(32095.165)', 'np.float64(34004.52)', 'np.float64(36092.745)', 'np.float64(38102.757)', 'np.float64(40279.639)', 'np.float64(42528.336)', 'np.float64(44922.49)', 'np.float64(47644.187)', 'np.float64(50393.853)', 'np.float64(53103.223)']
  row 10: ["'Changes in debt held by the pub

In [21]:
# %% Cell 5 — Parse Table 1-3 by explicit row index
def parse_table_1_3(xlsx_path):
    raw = pd.read_excel(xlsx_path, sheet_name="Table 1-3", header=None)

    # Row 7: year headers — first col is "Actual, \n2025" (not a bare year), rest are floats
    col_years = {}
    for ci, v in enumerate(raw.iloc[7]):
        c = clean(v)
        if c is None:
            continue
        # Handle "Actual, \n2025" specially
        if "2025" in c:
            col_years[ci] = 2025
            continue
        if is_year(v):
            col_years[ci] = int(float(c))

    def get_row(row_idx):
        row = raw.iloc[row_idx]
        return {yr: to_num(row.iloc[ci]) for ci, yr in col_years.items()}

    data = {
        "Debt at Beginning of Year":     get_row(8),
        "Change from Deficit":           get_row(11),
        "Change from Other Financing":   get_row(12),
        "Debt at End of Year (Billions)": get_row(16),
        "Debt at End of Year (% GDP)":   get_row(17),
        "Federal Financial Assets":      get_row(20),
    }

    return sorted(col_years.values()), data

years_13, data_13 = parse_table_1_3(XLSX)

print(f"Years: {years_13}")
print("\nSpot checks (2025 / 2026):")
checks_13 = {
    "Debt at Beginning of Year":      (28195.575, 30172.402),
    "Change from Deficit":            (1775.357,  1852.703),
    "Debt at End of Year (Billions)": (30172.402, 32095.165),
    "Debt at End of Year (% GDP)":    (99.375,    100.605),
}
for label, (exp25, exp26) in checks_13.items():
    v25 = data_13[label][2025]
    v26 = data_13[label][2026]
    ok25 = "✓" if abs(v25 - exp25) < 0.01 else "✗"
    ok26 = "✓" if abs(v26 - exp26) < 0.01 else "✗"
    print(f"  {label:35s} 2025: {v25:.3f} {ok25}  2026: {v26:.3f} {ok26}")


Years: [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036]

Spot checks (2025 / 2026):
  Debt at Beginning of Year           2025: 28195.575 ✓  2026: 30172.402 ✓
  Change from Deficit                 2025: 1775.357 ✓  2026: 1852.703 ✓
  Debt at End of Year (Billions)      2025: 30172.402 ✓  2026: 32095.165 ✓
  Debt at End of Year (% GDP)         2025: 99.375 ✓  2026: 100.605 ✓


In [30]:
# %% Cell 6 — Parse Table 5-1 by explicit row index
def parse_table_5_1(xlsx_path):
    raw = pd.read_excel(xlsx_path, sheet_name="Table 5-1", header=None)

    # Row 7: year headers 2026-2035 only (no 2025, no 2036); skip summary cols
    col_years = {}
    for ci, v in enumerate(raw.iloc[7]):
        if is_year(v):
            col_years[ci] = int(float(clean(v)))

    def get_row(row_idx):
        row = raw.iloc[row_idx]
        return {yr: to_num(row.iloc[ci]) for ci, yr in col_years.items()}

    data = {
        "Jan 2025 Baseline Deficit":         get_row(9),    # pre-OBBBA
        "Feb 2026 Baseline Deficit":         get_row(113),  # current law
        "Net Interest Change Legislative":   get_row(49),
        "Net Interest Change Economic":      get_row(75),
        "Net Interest Change Technical":     get_row(107),
    }

    return sorted(col_years.values()), data

years_51, data_51 = parse_table_5_1(XLSX)

print(f"Years: {years_51}")
print("\nSpot checks:")
checks_51 = {
    "Jan 2025 Baseline Deficit":       {2026: 1713.258, 2027: 1686.942, 2035: 2531.378},
    "Feb 2026 Baseline Deficit":       {2026: 1852.703, 2027: 1887.203, 2035: 2779.034},
    "Net Interest Change Legislative": {2026: 4.696,    2027: 27.514,   2035: 121.622},
    "Net Interest Change Economic":    {2026: 14.543,   2027: 11.248,   2035: 210.593},
    "Net Interest Change Technical":   {2026: 9.461,    2027: -6.285,   2035: -95.728},
}
for label, expected in checks_51.items():
    print(f"  {label}:")
    for yr, exp in expected.items():
        v = data_51[label][yr]
        ok = "✓" if abs(v - exp) < 0.01 else "✗"
        print(f"    {yr}: {v:.3f}  (expect {exp}) {ok}")



Years: [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035]

Spot checks:
  Jan 2025 Baseline Deficit:
    2026: 1713.258  (expect 1713.258) ✓
    2027: 1686.942  (expect 1686.942) ✓
    2035: 2531.378  (expect 2531.378) ✓
  Feb 2026 Baseline Deficit:
    2026: 1852.703  (expect 1852.703) ✓
    2027: 1887.203  (expect 1887.203) ✓
    2035: 2779.034  (expect 2779.034) ✓
  Net Interest Change Legislative:
    2026: 4.696  (expect 4.696) ✓
    2027: 27.514  (expect 27.514) ✓
    2035: 121.622  (expect 121.622) ✓
  Net Interest Change Economic:
    2026: 14.543  (expect 14.543) ✓
    2027: 11.248  (expect 11.248) ✓
    2035: 210.593  (expect 210.593) ✓
  Net Interest Change Technical:
    2026: 9.461  (expect 9.461) ✓
    2027: -6.285  (expect -6.285) ✓
    2035: -95.728  (expect -95.728) ✓


In [31]:
# %% Cell 7 — Build projections_deficit.csv (scenarios A & B)
rows = []

# Scenario A: Jan 2025 baseline (pre-OBBBA), years 2026-2035
for yr, v in sorted(data_51["Jan 2025 Baseline Deficit"].items()):
    if v is not None:
        rows.append({"scenario": "jan_2025_baseline", "year": yr, "deficit_billions": round(v, 1)})

# Scenario B: Feb 2026 current law (from Table 1-1, abs value, years 2025-2036)
for yr in sorted(years_11):
    v = data_11["Total Deficit"].get(yr)
    if v is not None:
        rows.append({"scenario": "feb_2026_current_law", "year": yr, "deficit_billions": round(abs(v), 1)})

df_deficit = pd.DataFrame(rows).sort_values(["scenario", "year"])

print(df_deficit.pivot(index="year", columns="scenario", values="deficit_billions").to_string())

print("\n10-year totals (2026-2035):")
for sc in ["jan_2025_baseline", "feb_2026_current_law"]:
    total = df_deficit[(df_deficit.scenario == sc) & df_deficit.year.between(2026, 2035)]["deficit_billions"].sum()
    print(f"  {sc:25s}: ${total:,.0f}B")
print("  Expected: jan ~$21,758B  |  feb ~$23,143B")

scenario  feb_2026_current_law  jan_2025_baseline
year                                             
2025                    1775.4                NaN
2026                    1852.7             1713.3
2027                    1887.2             1686.9
2028                    2079.7             1910.8
2029                    2019.7             1938.3
2030                    2200.6             2139.7
2031                    2285.7             2232.8
2032                    2439.4             2370.6
2033                    2780.7             2637.3
2034                    2818.5             2597.2
2035                    2779.0             2531.4
2036                    3115.4                NaN

10-year totals (2026-2035):
  jan_2025_baseline        : $21,758B
  feb_2026_current_law     : $23,143B
  Expected: jan ~$21,758B  |  feb ~$23,143B


In [32]:
# %% Cell 8 — Derive Jan 2025 net interest baseline
# Feb 2026 net interest (Table 1-1) minus all three change components (Table 5-1)
# = Jan 2025 net interest level

# Table 5-1 only covers 2026-2035; we have Feb 2026 for 2025 and 2036 too
rows_ni = []

# Feb 2026 net interest — all years from Table 1-1
for yr in sorted(years_11):
    v = data_11["Net Interest"].get(yr)
    if v is not None:
        rows_ni.append({"scenario": "feb_2026_current_law", "year": yr, "net_interest_billions": round(v, 1)})

# Jan 2025 net interest — back out changes for 2026-2035 only
for yr in sorted(years_51):
    feb  = data_11["Net Interest"].get(yr)
    leg  = data_51["Net Interest Change Legislative"].get(yr)
    econ = data_51["Net Interest Change Economic"].get(yr)
    tech = data_51["Net Interest Change Technical"].get(yr)
    if all(v is not None for v in [feb, leg, econ, tech]):
        jan = feb - leg - econ - tech
        rows_ni.append({"scenario": "jan_2025_baseline", "year": yr, "net_interest_billions": round(jan, 1)})

df_ni = pd.DataFrame(rows_ni).sort_values(["scenario", "year"])

print(df_ni.pivot(index="year", columns="scenario", values="net_interest_billions").to_string())

scenario  feb_2026_current_law  jan_2025_baseline
year                                             
2025                     969.9                NaN
2026                    1039.0             1010.3
2027                    1107.7             1075.2
2028                    1217.8             1164.5
2029                    1324.5             1247.2
2030                    1431.8             1327.5
2031                    1548.4             1417.0
2032                    1670.4             1513.9
2033                    1784.4             1604.5
2034                    1903.8             1693.6
2035                    2019.1             1782.6
2036                    2144.3                NaN


In [33]:
# %% Cell 9 — Write output CSVs

# 1. projections_deficit.csv
df_deficit.to_csv(OUT_DIR / "projections_deficit.csv", index=False)
print(f"Wrote projections_deficit.csv ({len(df_deficit)} rows)")

# 2. projections_net_interest.csv
df_ni.to_csv(OUT_DIR / "projections_net_interest.csv", index=False)
print(f"Wrote projections_net_interest.csv ({len(df_ni)} rows)")

# 3. projections_summary.csv — key annual series from Table 1-1 and 1-3
summary_rows = []
cats_11 = {
    "Total Revenues":        "Total Revenues",
    "Total Outlays":         "Total Outlays",
    "Mandatory Outlays":     "Mandatory Outlays",
    "Discretionary Outlays": "Discretionary Outlays",
    "Net Interest":          "Net Interest",
    "Debt Held by Public":   "Debt Held by Public",
    "GDP":                   "GDP",
}
for label, key in cats_11.items():
    for yr in sorted(years_11):
        v = data_11[key].get(yr)
        if v is not None:
            summary_rows.append({"category": label, "year": yr, "amount_billions": round(v, 1)})

# Add debt % GDP from Table 1-3
for yr in sorted(years_13):
    v = data_13["Debt at End of Year (% GDP)"].get(yr)
    if v is not None:
        summary_rows.append({"category": "Debt Held by Public (% GDP)", "year": yr, "amount_billions": round(v, 3)})

df_summary = pd.DataFrame(summary_rows).sort_values(["category", "year"])
df_summary.to_csv(OUT_DIR / "projections_summary.csv", index=False)
print(f"Wrote projections_summary.csv ({len(df_summary)} rows)")

print("\nAll done. Files in:", OUT_DIR.resolve())

rows = []

# Scenario A: Jan 2025 baseline (pre-OBBBA), years 2026-2035
for yr, v in sorted(data_51["Jan 2025 Baseline Deficit"].items()):
    if v is not None:
        rows.append({"scenario": "jan_2025_baseline", "year": yr, "deficit_billions": round(v, 1)})

# Scenario B: Feb 2026 current law (from Table 1-1, abs value, years 2025-2036)
for yr in sorted(years_11):
    v = data_11["Total Deficit"].get(yr)
    if v is not None:
        rows.append({"scenario": "feb_2026_current_law", "year": yr, "deficit_billions": round(abs(v), 1)})

df_deficit = pd.DataFrame(rows).sort_values(["scenario", "year"])

print(df_deficit.pivot(index="year", columns="scenario", values="deficit_billions").to_string())

print("\n10-year totals (2026-2035):")
for sc in ["jan_2025_baseline", "feb_2026_current_law"]:
    total = df_deficit[(df_deficit.scenario == sc) & df_deficit.year.between(2026, 2035)]["deficit_billions"].sum()
    print(f"  {sc:25s}: ${total:,.0f}B")
print("  Expected: jan ~$21,758B  |  feb ~$23,143B")

Wrote projections_deficit.csv (22 rows)
Wrote projections_net_interest.csv (22 rows)
Wrote projections_summary.csv (96 rows)

All done. Files in: /Users/aaditbhatia/Desktop/OBBB Dashboard/viz/public
scenario  feb_2026_current_law  jan_2025_baseline
year                                             
2025                    1775.4                NaN
2026                    1852.7             1713.3
2027                    1887.2             1686.9
2028                    2079.7             1910.8
2029                    2019.7             1938.3
2030                    2200.6             2139.7
2031                    2285.7             2232.8
2032                    2439.4             2370.6
2033                    2780.7             2637.3
2034                    2818.5             2597.2
2035                    2779.0             2531.4
2036                    3115.4                NaN

10-year totals (2026-2035):
  jan_2025_baseline        : $21,758B
  feb_2026_current_law     : $23,1